# 15 — Machine Learning with Streamlit

## 📓 Interactive Notebook · Module 11 · ML

In this notebook, you'll learn:
1. **Model persistence** with joblib
2. **Preprocessing consistency** between training and inference
3. **Building classification apps**
4. **Building regression apps**
5. **Handling invalid inputs**
6. **Batch predictions**

---

## 📋 Objectives

By the end of this notebook, you will be able to:
- Save and load trained models with joblib
- Create preprocessing pipelines that match training
- Build interactive ML prediction apps
- Handle invalid inputs gracefully
- Implement batch prediction workflows

## 📋 Prerequisites

- Modules 01–10 completed
- Basic ML knowledge (train/test split, fit/predict)
- Scikit-learn familiarity

---

## 💡 The ML Deployment Pipeline

```
TRAINING (Offline)          DEPLOYMENT (Streamlit)
─────────────────           ─────────────────────
1. Load Data                1. Load Model
2. Preprocess               2. Get User Input
3. Train Model              3. Preprocess Input
4. Save Model       →      4. Predict
5. Save Preprocessor        5. Display Results
```

**Key Principle:** Train once, predict many times.

---

## 💡 Step 1: Train and Save a Model

First, we train a model and save it for later use.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
import joblib

# Load sample dataset
iris = load_iris()
X = pd.DataFrame(iris.data, columns=iris.feature_names)
y = iris.target
class_names = iris.target_names

st.header("📝 Step 1: Train and Save Model")
st.write("**Dataset:** Iris (150 samples, 4 features, 3 classes)")
st.dataframe(X.head())

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Preprocessing
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)  # Only transform!

# Train model
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train_scaled, y_train)

# Evaluate
accuracy = model.score(X_test_scaled, y_test)
st.write(f"**Test Accuracy:** {accuracy:.2%}")

# Save artifacts
joblib.dump(model, "iris_model.joblib")
joblib.dump(scaler, "iris_scaler.joblib")

st.success("✅ Model and scaler saved!")

---

## 💡 Step 2: Load Model in Streamlit

Use `@st.cache_resource` to cache the model.

In [ ]:
import streamlit as st
import joblib

st.header("📝 Step 2: Load Model")

@st.cache_resource
def load_model():
    """Load model and scaler (cached as singletons)."""
    model = joblib.load("iris_model.joblib")
    scaler = joblib.load("iris_scaler.joblib")
    return model, scaler

model, scaler = load_model()

st.write(f"**Model type:** {type(model).__name__}")
st.write(f"**Scaler type:** {type(scaler).__name__}")
st.write(f"**Feature names:** {iris.feature_names}")

st.info("💡 Model is cached — loading is instant on subsequent calls.")

---

## 💡 Step 3: Feature Input UI

Create widgets for each feature.

In [ ]:
import streamlit as st
import numpy as np

st.header("📝 Step 3: Feature Input UI")

# Input widgets for each feature
col1, col2 = st.columns(2)

with col1:
    sepal_length = st.slider(
        "Sepal Length (cm)",
        min_value=4.0, max_value=8.0, value=5.4, step=0.1
    )
    sepal_width = st.slider(
        "Sepal Width (cm)",
        min_value=2.0, max_value=4.5, value=3.4, step=0.1
    )

with col2:
    petal_length = st.slider(
        "Petal Length (cm)",
        min_value=1.0, max_value=7.0, value=1.5, step=0.1
    )
    petal_width = st.slider(
        "Petal Width (cm)",
        min_value=0.1, max_value=2.5, value=0.2, step=0.1
    )

# Create feature array
features = np.array([[sepal_length, sepal_width, petal_length, petal_width]])

st.write("**Input features:**")
st.json({
    "sepal_length": sepal_length,
    "sepal_width": sepal_width,
    "petal_length": petal_length,
    "petal_width": petal_width
})

---

## 💡 Step 4: Make Predictions

Preprocess input and predict.

In [ ]:
import streamlit as st
import numpy as np

st.header("📝 Step 4: Predict")

# Input (using values from Step 3)
col1, col2, col3, col4 = st.columns(4)
with col1:
    sl = st.slider("Sepal Length", 4.0, 8.0, 5.4, 0.1, key="sl")
with col2:
    sw = st.slider("Sepal Width", 2.0, 4.5, 3.4, 0.1, key="sw")
with col3:
    pl = st.slider("Petal Length", 1.0, 7.0, 1.5, 0.1, key="pl")
with col4:
    pw = st.slider("Petal Width", 0.1, 2.5, 0.2, 0.1, key="pw")

features = np.array([[sl, sw, pl, pw]])

if st.button("Predict Species"):
    # CRITICAL: Preprocess EXACTLY like training
    features_scaled = scaler.transform(features)  # transform only!
    
    # Predict
    prediction = model.predict(features_scaled)[0]
    probabilities = model.predict_proba(features_scaled)[0]
    
    # Display results
    st.success(f"**Predicted Species:** {class_names[prediction]}")
    
    # Show probabilities
    st.write("**Confidence:**")
    for name, prob in zip(class_names, probabilities):
        st.progress(prob, text=f"{name}: {prob:.1%}")
    
    st.info("💡 Note: transform() not fit_transform() — must match training!")

---

## ⚠️ Preprocessing Consistency

**The #1 mistake in ML deployment:** preprocessing mismatch.

In [ ]:
import streamlit as st

st.header("⚠️ Preprocessing Consistency")

st.subheader("❌ WRONG: Fitting at inference time")
st.code('''
# NEVER do this!
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)  # fit_transform at inference!
prediction = model.predict(X_scaled)
# Problem: scaler was fit on different data than training!
''', language="python")

st.subheader("✅ CORRECT: Transform only at inference")
st.code('''
# CORRECT approach
model = joblib.load("model.joblib")
scaler = joblib.load("scaler.joblib")  # Already fitted during training

X_scaled = scaler.transform(X)  # transform only!
prediction = model.predict(X_scaled)
# Scaler uses same statistics as training
''', language="python")

st.warning("⚠️ Always save your scaler/encoder alongside your model!")

---

## 💡 Invalid Input Handling

Validate inputs before prediction.

In [ ]:
import streamlit as st
import numpy as np

st.header("💡 Invalid Input Handling")

def validate_iris_input(sl, sw, pl, pw):
    """Validate Iris inputs."""
    errors = []
    
    if not (4.0 <= sl <= 8.0):
        errors.append("Sepal Length must be 4.0-8.0 cm")
    if not (2.0 <= sw <= 4.5):
        errors.append("Sepal Width must be 2.0-4.5 cm")
    if not (1.0 <= pl <= 7.0):
        errors.append("Petal Length must be 1.0-7.0 cm")
    if not (0.1 <= pw <= 2.5):
        errors.append("Petal Width must be 0.1-2.5 cm")
    
    return errors

# Test with sliders
sl = st.slider("Sepal Length", 3.0, 9.0, 5.4, 0.1, key="val_sl")
sw = st.slider("Sepal Width", 1.5, 5.0, 3.4, 0.1, key="val_sw")
pl = st.slider("Petal Length", 0.5, 7.5, 1.5, 0.1, key="val_pl")
pw = st.slider("Petal Width", 0.0, 3.0, 0.2, 0.1, key="val_pw")

if st.button("Validate & Predict"):
    errors = validate_iris_input(sl, sw, pl, pw)
    
    if errors:
        for error in errors:
            st.error(f"❌ {error}")
    else:
        features = np.array([[sl, sw, pl, pw]])
        features_scaled = scaler.transform(features)
        prediction = model.predict(features_scaled)[0]
        st.success(f"✅ Prediction: {class_names[prediction]}")

---

## 💡 Batch Prediction

Predict on multiple samples from uploaded file.

In [ ]:
import streamlit as st
import pandas as pd
import numpy as np

st.header("💡 Batch Prediction")

# Create sample CSV for download
sample_data = pd.DataFrame({
    "sepal_length": [5.1, 4.9, 7.0, 6.4],
    "sepal_width": [3.5, 3.0, 3.2, 3.2],
    "petal_length": [1.4, 1.4, 4.7, 4.5],
    "petal_width": [0.2, 0.2, 1.4, 1.5]
})
csv = sample_data.to_csv(index=False)
st.download_button("📥 Download Sample CSV", csv, "sample_iris.csv", "text/csv")

# Upload file
uploaded = st.file_uploader("Upload CSV for batch prediction", type=["csv"])

if uploaded:
    df = pd.read_csv(uploaded)
    st.write("**Uploaded data:**")
    st.dataframe(df.head())
    
    # Validate columns
    required = ["sepal_length", "sepal_width", "petal_length", "petal_width"]
    missing = [col for col in required if col not in df.columns]
    
    if missing:
        st.error(f"Missing columns: {', '.join(missing)}")
    else:
        # Handle missing values
        if df[required].isnull().any().any():
            st.warning("Filling missing values with median.")
            df[required] = df[required].fillna(df[required].median())
        
        # Preprocess
        X = df[required].values
        X_scaled = scaler.transform(X)
        
        # Predict
        predictions = model.predict(X_scaled)
        probabilities = model.predict_proba(X_scaled)
        
        # Add results
        df["predicted_species"] = [class_names[p] for p in predictions]
        df["confidence"] = probabilities.max(axis=1)
        
        st.success(f"✅ Generated {len(predictions)} predictions")
        st.dataframe(df)
        
        # Download
        csv = df.to_csv(index=False)
        st.download_button("📥 Download Results", csv, "predictions.csv", "text/csv")

---

## 🎯 Regression Example

Apply the same pattern to regression.

In [ ]:
import streamlit as st
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import GradientBoostingRegressor
import joblib

st.header("🎯 Regression Example: California Housing")

# Train regression model
housing = fetch_california_housing()
X = pd.DataFrame(housing.data, columns=housing.feature_names)
y = housing.target  # Median house value (in $100k)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler_reg = StandardScaler()
X_train_scaled = scaler_reg.fit_transform(X_train)
X_test_scaled = scaler_reg.transform(X_test)

model_reg = GradientBoostingRegressor(n_estimators=100, random_state=42)
model_reg.fit(X_train_scaled, y_train)

r2_train = model_reg.score(X_train_scaled, y_train)
r2_test = model_reg.score(X_test_scaled, y_test)

st.write(f"**Train R²:** {r2_train:.3f}")
st.write(f"**Test R²:** {r2_test:.3f}")

# Save
joblib.dump(model_reg, "housing_model.joblib")
joblib.dump(scaler_reg, "housing_scaler.joblib")

st.success("✅ Regression model saved!")

# Prediction UI
st.subheader("Predict House Price")

col1, col2 = st.columns(2)
with col1:
    med_income = st.slider("Median Income ($10k)", 0.0, 15.0, 3.0, 0.1)
    house_age = st.slider("House Age", 0.0, 50.0, 10.0, 1.0)
with col2:
    avg_rooms = st.slider("Avg Rooms", 0.0, 10.0, 5.0, 0.5)
    avg_bedrooms = st.slider("Avg Bedrooms", 0.0, 3.0, 1.0, 0.1)

if st.button("Predict Price"):
    # Use remaining features with defaults
    features = np.array([[
        avg_rooms, avg_bedrooms, house_age,
        0.0, 0.0, 35.0, -120.0, med_income
    ]])
    features_scaled = scaler_reg.transform(features)
    prediction = model_reg.predict(features_scaled)[0]
    
    st.success(f"💰 Predicted Price: ${prediction * 100000:,.0f}")

---

## ⚠️ Common Mistakes

### Mistake 1: Training on Every Rerun
```python
# ❌ WRONG
model = train_model(data)  # Slow!

# ✅ CORRECT
model = joblib.load("model.joblib")  # Fast!
```

### Mistake 2: Preprocessing Mismatch
```python
# ❌ WRONG
scaler.fit_transform(X)  # fit_transform at inference!

# ✅ CORRECT
scaler.transform(X)  # transform only
```

### Mistake 3: Not Saving Preprocessor
```python
# ❌ WRONG
joblib.dump(model, "model.joblib")  # Where's the scaler?

# ✅ CORRECT
joblib.dump(model, "model.joblib")
joblib.dump(scaler, "scaler.joblib")  # Save both!
```

---

## 🎯 Challenges

### Challenge 1: Complete Classification App
Build a full classification app with:
- Input validation
- Confidence display
- Error handling

### Challenge 2: Regression with Confidence
Extend regression to show prediction intervals.

### Challenge 3: Multi-Model Selector
Let users choose between different trained models.

In [ ]:
# Challenge 1: Complete Classification App
import streamlit as st

st.write("TODO: Build a complete classification app")

# Your code here


---

## 📝 Key Takeaways

1. **Train offline, load in app** — never train on every rerun

2. **Cache models with `@st.cache_resource`** — singletons for performance

3. **Save preprocessors alongside models** — scaler, encoder, feature names

4. **Preprocessing must match exactly** — `transform()`, not `fit_transform()`

5. **Validate all inputs** — ranges, types, required fields

6. **Show confidence** — probabilities for classification, intervals for regression

7. **Handle edge cases** — missing data, invalid inputs, model failures

---

## 📚 Further Reading

- [Scikit-learn Model Persistence](https://scikit-learn.org/stable/model_persistence.html)
- [Joblib Documentation](https://joblib.readthedocs.io/)

---

## 🔗 Related Materials

- 📖 Reading: [15 — ML with Streamlit](../readings/15_machine_learning_streamlit.md)
- ✏️ Exercise: [15 — ML Workshop](../exercises/15_ml_workshop.py)
- 🖥️ Demo App: [15 — Classification App](../apps/15_classification_app.py)
- 🖥️ Demo App: [15 — Regression App](../apps/15_regression_app.py)
- 📝 Quiz: [11 — Machine Learning](../quizzes/11_machine_learning.md)
- 🚀 Project: [P06 — ML Model Playground](../projects/P06_ml_model_playground.md)